# Chinese world — catalogs ↔ main polities (Sankey flow)

Counterpart to **Laouenan et al. 2022, Figure S7**: relation between external catalogs (left) and Chinese-world polities (right). Each ribbon's width = number of *distinct individuals* of a given polity that carry an identifier in the catalog.

Catalogs are external biographical / authority databases (CBDB, Shanghai Library, ctext, Academia Sinica, VIAF, GND, …) — Wikipedia / Wikidata language editions are excluded so the left side is genuinely about *third-party* catalog coverage, not about Wikipedia itself.

Polities on the right are the same 12 main polities used in `fig_chinese_polities_share.ipynb` and grouped from the canonical list in `polities_cliopatria` (no invented names or dates).

## 1. Configuration

In [1]:
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DB_PATH = '../data/humans_clean.duckdb'

# Top-N external catalogs to keep on the left (rest pooled into 'Other catalogs').
TOP_CATALOGS = 12

# Drop ribbons smaller than this many individuals (keeps the figure readable).
MIN_FLOW = 5

# Polity palette — matches `paper/fig_chinese_polities_share.ipynb`.
PALETTE = {
    'Shang':          '#6c2a2c',
    'Zhou':           '#8e4f33',
    'Qin':            '#b07a3f',
    'Han':            '#c8a472',
    'Six Dynasties':  '#8e7a8a',
    'Sui':            '#6c8a8a',
    'Tang':           '#4f8a98',
    'Five Dynasties': '#5b6f8e',
    'Song':           '#3e5b8e',
    'Yuan':           '#6a4f8a',
    'Ming':           '#8e4f7a',
    'Qing':           '#a05a6a',
}

# Catalog node colour (neutral, restrained).
CATALOG_COLOR = '#4a4a4a'

POLITY_TO_MAIN = {
    'Shang Dynasty':                   'Shang',
    'Zhou Dynasty':                    'Zhou',
    'Qin Dynasty':                     'Qin',
    'Han Dynasty':                     'Han',
    'Xin Dynasty':                     'Han',
    'Western Jin':                     'Six Dynasties',
    'Liu Song Dynasty':                'Six Dynasties',
    'Liang Dynasty':                   'Six Dynasties',
    'Chen Dynasty':                    'Six Dynasties',
    'Northern Wei':                    'Six Dynasties',
    'Eastern Wei':                     'Six Dynasties',
    'Western Wei':                     'Six Dynasties',
    'Northern Zhou':                   'Six Dynasties',
    'Northern Qi':                     'Six Dynasties',
    'Sui Dynasty':                     'Sui',
    'Tang Dynasty':                    'Tang',
    'Five Dynasties and Ten Kingdoms': 'Five Dynasties',
    'Liao Dynasty':                    'Five Dynasties',
    'Western Xia':                     'Five Dynasties',
    'Northern Song':                   'Song',
    'Southern Song':                   'Song',
    'Yuan Dynasty':                    'Yuan',
    'Ming Dynasty':                    'Ming',
    'Qing Dynasty':                    'Qing',
}

# Plot order = chronological order on the right side.
MAIN_ORDER = ['Shang', 'Zhou', 'Qin', 'Han', 'Six Dynasties', 'Sui',
              'Tang', 'Five Dynasties', 'Song', 'Yuan', 'Ming', 'Qing']

In [2]:
import duckdb
import polars as pl
import plotly.graph_objects as go

## 2. Load (individual, catalog, polity) triples

Pull every `(wikidata_id, identifier_name, polity_name)` triple where the individual is attached to one of the canonical Chinese polities. We then map `polity_name` → main polity and exclude Wikipedia / Wikidata identifiers (which would dwarf real catalogs).

In [3]:
polities = list(POLITY_TO_MAIN)
ph = ','.join(['?'] * len(polities))

conn = duckdb.connect(DB_PATH, read_only=True)
triples = conn.execute(f"""
    SELECT DISTINCT i.wikidata_id,
           i.identifier_name AS catalog,
           ic.polity_name    AS polity
    FROM identifiers i
    JOIN individuals_cliopatria ic USING (wikidata_id)
    WHERE ic.polity_name IN ({ph})
      AND i.identifier_name IS NOT NULL
      AND i.identifier_name NOT ILIKE '%Wikipedia%'
      AND i.identifier_name NOT ILIKE '%Wikidata%'
      AND i.identifier_name NOT ILIKE '%Wikimedia%'
      AND i.identifier_name NOT ILIKE '%Wikisource%'
""", polities).pl()
conn.close()

triples = triples.with_columns(
    pl.col('polity').replace_strict(POLITY_TO_MAIN).alias('main')
).drop('polity')

print(f'{triples.height:,} (individual, catalog, main) triples')
print(f'unique individuals: {triples["wikidata_id"].n_unique():,}')
print(f'unique catalogs:    {triples["catalog"].n_unique():,}')
triples.head()

277,549 (individual, catalog, main) triples
unique individuals: 84,986
unique catalogs:    631


wikidata_id,catalog,main
str,str,str
"""Q887588""","""Swedish Open Cultural Heritage…","""Qing"""
"""Q127017""","""Union List of Artist Names ID""","""Qing"""
"""Q127017""","""SNAC ARK ID""","""Qing"""
"""Q507434""","""Biografija.ru ID""","""Qing"""
"""Q228889""","""CONOR.SI ID""","""Ming"""


## 3. Top catalogs and pooling

Rank catalogs by total number of distinct Chinese-world individuals they cover and keep the top-N; everything else is pooled as **Other catalogs**.

In [4]:
cat_ranks = (
    triples.group_by('catalog')
    .agg(pl.col('wikidata_id').n_unique().alias('n_indiv'))
    .sort('n_indiv', descending=True)
)
top = cat_ranks.head(TOP_CATALOGS)['catalog'].to_list()
print('Top catalogs:')
print(cat_ranks.head(TOP_CATALOGS + 5))

triples = triples.with_columns(
    pl.when(pl.col('catalog').is_in(top))
    .then(pl.col('catalog'))
    .otherwise(pl.lit('Other catalogs'))
    .alias('catalog_grp')
)

# Final order on the left: top catalogs descending, then 'Other catalogs'.
CATALOG_ORDER = top + ['Other catalogs']

Top catalogs:
shape: (17, 2)
┌─────────────────────────────────┬─────────┐
│ catalog                         ┆ n_indiv │
│ ---                             ┆ ---     │
│ str                             ┆ u32     │
╞═════════════════════════════════╪═════════╡
│ CBDB ID                         ┆ 80127   │
│ Shanghai Library person ID      ┆ 74221   │
│ Google Knowledge Graph ID       ┆ 19308   │
│ Geni.com profile ID             ┆ 14793   │
│ ctext data entity ID            ┆ 13378   │
│ …                               ┆ …       │
│ Modern History Database person… ┆ 2529    │
│ HKCAN ID                        ┆ 2520    │
│ NLA Trove people ID             ┆ 2026    │
│ GND ID                          ┆ 2001    │
│ Encyclopedia of China (Third E… ┆ 1922    │
└─────────────────────────────────┴─────────┘


## 4. Build flows (catalog → main polity)

Flow value = number of *distinct individuals* of polity `m` that carry at least one identifier of catalog `c`. Tiny ribbons (`< MIN_FLOW`) are dropped — they are visual noise, the unfiltered counts are still in the summary table at the bottom.

In [5]:
flows = (
    triples.select(['wikidata_id', 'catalog_grp', 'main']).unique()
    .group_by(['catalog_grp', 'main'])
    .agg(pl.col('wikidata_id').n_unique().alias('value'))
    .filter(pl.col('value') >= MIN_FLOW)
    .sort(['catalog_grp', 'main'])
)

print(f'{flows.height} flows after pruning ribbons < {MIN_FLOW}')
flows.head(15)

124 flows after pruning ribbons < 5


catalog_grp,main,value
str,str,u32
"""Academia Sinica authority ID""","""Han""",5
"""Academia Sinica authority ID""","""Ming""",3302
"""Academia Sinica authority ID""","""Qing""",8206
"""Academia Sinica authority ID""","""Song""",31
"""Academia Sinica authority ID""","""Tang""",11
…,…,…
"""CBDB ID""","""Six Dynasties""",507
"""CBDB ID""","""Song""",6766
"""CBDB ID""","""Sui""",1003


## 5. Figure — Sankey: catalogs (left) → main Chinese polities (right)

Node positions are pinned: catalogs stack on the left in descending coverage; polities stack on the right in chronological order (Shang at the top → Qing at the bottom). Ribbons are tinted by the destination polity, matching the colour key of `fig_chinese_polities_share.ipynb`.

In [6]:
def hex_to_rgba(hex_color, alpha):
    h = hex_color.lstrip('#')
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f'rgba({r},{g},{b},{alpha})'

node_labels = list(CATALOG_ORDER) + list(MAIN_ORDER)
node_colors = ([CATALOG_COLOR] * len(CATALOG_ORDER)
               + [PALETTE[m] for m in MAIN_ORDER])

cat_idx  = {c: i for i, c in enumerate(CATALOG_ORDER)}
main_idx = {m: len(CATALOG_ORDER) + i for i, m in enumerate(MAIN_ORDER)}

src = [cat_idx[c]  for c in flows['catalog_grp']]
tgt = [main_idx[m] for m in flows['main']]
val = flows['value'].to_list()
link_colors = [hex_to_rgba(PALETTE[m], 0.45) for m in flows['main']]

# Pin node positions. Plotly normalises x,y to [0,1].
n_cat  = len(CATALOG_ORDER)
n_main = len(MAIN_ORDER)
x_cat  = [0.01] * n_cat
y_cat  = [(i + 0.5) / n_cat  for i in range(n_cat)]
x_main = [0.99] * n_main
y_main = [(i + 0.5) / n_main for i in range(n_main)]

fig = go.Figure(data=[go.Sankey(
    arrangement='fixed',
    node=dict(
        pad=14,
        thickness=18,
        line=dict(color='white', width=0.5),
        label=node_labels,
        color=node_colors,
        x=x_cat + x_main,
        y=y_cat + y_main,
        hovertemplate='%{label}<br>%{value:,} individuals<extra></extra>',
    ),
    link=dict(
        source=src,
        target=tgt,
        value=val,
        color=link_colors,
        hovertemplate=('%{source.label} → %{target.label}'
                       '<br>%{value:,} individuals<extra></extra>'),
    ),
)])

fig.update_layout(
    font=dict(family='DejaVu Sans', size=13, color='#222'),
    paper_bgcolor='white',
    plot_bgcolor='white',
    margin=dict(l=10, r=10, t=20, b=10),
    width=1100,
    height=620,
)
fig.show()

## 6. Summary tables

Two views of the same data:

1. **per polity** — total catalog coverage and which catalog carries the most individuals,
2. **per catalog** — how each catalog's load is split across polities.

In [7]:
by_polity = (
    triples.select(['wikidata_id', 'main']).unique()
    .group_by('main')
    .agg(pl.col('wikidata_id').n_unique().alias('individuals'))
    .with_columns(
        pl.col('main').replace_strict({n: i for i, n in enumerate(MAIN_ORDER)}).alias('_o')
    )
    .sort('_o')
    .drop('_o')
)
by_polity

main,individuals
str,u32
"""Shang""",7
"""Zhou""",11
"""Qin""",26
"""Han""",1067
"""Six Dynasties""",1274
…,…
"""Five Dynasties""",150
"""Song""",6845
"""Yuan""",2381


In [8]:
by_catalog = (
    triples.select(['wikidata_id', 'catalog_grp']).unique()
    .group_by('catalog_grp')
    .agg(pl.col('wikidata_id').n_unique().alias('individuals'))
    .with_columns(
        pl.col('catalog_grp').replace_strict({c: i for i, c in enumerate(CATALOG_ORDER)}).alias('_o')
    )
    .sort('_o')
    .drop('_o')
)
by_catalog

catalog_grp,individuals
str,u32
"""CBDB ID""",80127
"""Shanghai Library person ID""",74221
"""Google Knowledge Graph ID""",19308
"""Geni.com profile ID""",14793
"""ctext data entity ID""",13378
…,…
"""Library of Congress authority …",3257
"""Freebase ID""",3206
"""WorldCat Entities ID""",2637
